In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu129
%pip install transformers accelerate bitsandbytes langchain langchain-huggingface
%pip install pandas openai huggingface_hub

In [ ]:
import gc
import glob
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
import torch
from huggingface_hub import login
from langchain_core.runnables import RunnableSequence
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if 'failed' in gpu_info:
    print('Not connected to a GPU')
else:
    print(gpu_info)

print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
def majority_cohesion(group):
    majority = group['Cohesion_value'].mode()
    return majority.iloc[0] if len(majority) > 0 else group['Cohesion_value'].iloc[0]


def informativeness_in_majority(group):
    majority = group['Cohesion_value'].mode()
    majority_value = majority.iloc[0] if len(majority) > 0 else group['Cohesion_value'].iloc[0]

    if majority_value == 'cohesive':
        filtered = group[group['Cohesion_value'] == 'cohesive'].copy()
        filtered = filtered[filtered['Informativeness_value'].notnull()]

        if not filtered.empty:
            level_nums = filtered['Informativeness_value'].str.extract(r'(\d)', expand=False)
            level_nums = pd.to_numeric(level_nums, errors='coerce')
            max_idx = level_nums.idxmax()
            return filtered.loc[max_idx, 'Informativeness_value']
    return None


def aggregate_group(g):
    return {
        'source': g['source'].iloc[0],
        'target': g['target'].iloc[0],
        'Cohesion_value': majority_cohesion(g),
        'Informativeness_value': informativeness_in_majority(g),
    }

In [ ]:
with open("Task Guidelines shorter.yaml", "r") as f:
    guidelines = f.read()

In [ ]:
def normalize_output(text):
    text = text.strip()

    # Strip markdown code fences
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)
    text = text.strip()

    # Strategy 1: targeted regex for the expected JSON shape
    json_pattern = (
        r'\{\s*["\']cohesion["\']\s*:\s*["\']?(cohesive|incohesive)["\']?\s*,?\s*'
        r'["\']informativeness["\']\s*:\s*(\d+|null)\s*\}'
    )
    match = re.search(json_pattern, text, re.IGNORECASE)

    if match:
        coh = match.group(1).lower()
        info_str = match.group(2)
        info = int(info_str) if info_str != 'null' and info_str.isdigit() else None
    else:
        # Strategy 2: standard JSON parse on the first JSON object found
        short_text = text[:500]
        try:
            json_obj_match = re.search(r'\{[^}]+\}', short_text)
            if json_obj_match:
                parsed = json.loads(json_obj_match.group(0))
                coh = str(parsed.get("cohesion", "")).lower().strip()
                info = parsed.get("informativeness", None)
                if isinstance(info, str):
                    info = int(info) if info.isdigit() else None
            else:
                raise ValueError("No JSON found")
        except Exception:
            # Strategy 3: loose field-level regex fallback
            coh_match = re.search(r'"?cohesion"?\s*:\s*"?(cohesive|incohesive)"?', short_text, re.I)
            info_match = re.search(r'"?informativeness"?\s*:\s*(\d)', short_text)
            coh = coh_match.group(1).lower() if coh_match else "incohesive"
            info = int(info_match.group(1)) if info_match else None

    # Normalise cohesion label
    if coh and "cohes" in coh and "inco" not in coh:
        coh = "cohesive"
    else:
        coh = "incohesive"

    # Validate informativeness
    if info is not None:
        try:
            info = int(info)
            if info not in [1, 2, 3]:
                info = None
        except (ValueError, TypeError):
            info = None

    return {"cohesion": coh, "informativeness": info}

In [ ]:
def create_strict_prompt(context, elaboration, guidelines):
    return (
        f'CRITICAL: Respond with EXACTLY this JSON format and nothing else:\n'
        f'{{"cohesion":"cohesive","informativeness":2}}\n\n'
        f'Rules:\n'
        f'- "cohesion" must be "cohesive" or "incohesive"\n'
        f'- "informativeness" must be 1, 2, or 3 (or null if incohesive)\n'
        f'- NO explanations, NO extra text, NO code blocks\n\n'
        f'{guidelines}\n\n'
        f'Context: {context}\n\n'
        f'Elaboration: {elaboration}\n\n'
        f'JSON response:'
    )

In [ ]:
def create_json_prompt(context, elaboration, guidelines):
    """Build a chat-template CoT prompt for local HuggingFace models."""
    messages = [
        {
            "role": "system",
            "content": (
                f"{guidelines}\n\n"
                "EVALUATION CHECKLIST:\n"
                "1. Does the elaboration connect logically to the context?\n"
                "2. Does it maintain topic continuity without contradictions?\n"
                "3. Does it add new, accurate information?\n"
                "4. Is it free from repetition or errors?\n\n"
                "OUTPUT: Provide ONLY a JSON object.\n"
                'Format: {"cohesion":"cohesive","informativeness":2}\n'
                '- cohesion: "cohesive" or "incohesive"\n'
                "- informativeness: 1 (low), 2 (medium), 3 (high) or null if incohesive"
            ),
        },
        {
            "role": "user",
            "content": (
                f"Context: {context}\n\n"
                f"Elaboration: {elaboration}\n\n"
                "Evaluate using the checklist above. Return ONLY the JSON."
            ),
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
hf_models = [
    {"repo": "meta-llama/Llama-3.1-8B-Instruct",       "label": "llama-3.1-8b",  "model_kwargs": {"device_map": "auto"}},
    {"repo": "google/gemma-3-4b-it",                    "label": "gemma-3-4b-it", "model_kwargs": {"device_map": "auto"}},
    {"repo": "tiiuae/Falcon3-7B-Instruct",              "label": "falcon-3-7b",   "model_kwargs": {"device_map": "cuda"}},
    {"repo": "mistralai/Mistral-7B-Instruct-v0.1",      "label": "mistral-7b",    "model_kwargs": {"device_map": "auto"}},
]

openai_models = [
    {"name": "gpt-4o",       "label": "gpt-4o",       "temperature": 0.0, "max_tokens": 200},
    {"name": "gpt-4o-mini",  "label": "gpt-4o-mini",  "temperature": 0.0, "max_tokens": 200},
]

openrouter_models = [
    {"name": "google/gemini-2.5-pro",            "label": "gemini-2.5-pro",    "temperature": 0.3, "max_tokens": 500,
     "extra_body": {"reasoning": {"enabled": True}}},
    {"name": "qwen/qwen3-235b-a22b-instruct",    "label": "qwen3-235b",        "temperature": 0.0, "max_tokens": 200},
    {"name": "deepseek/deepseek-chat-v3-0324",   "label": "deepseek-chat-v3",  "temperature": 0.0, "max_tokens": 200},
]

In [ ]:
csv_files = sorted(glob.glob(os.path.join("Annotations", "*.csv")))
if not csv_files:
    raise FileNotFoundError("No CSV files found in Annotations/")

all_df = pd.concat([pd.read_csv(f, header=0) for f in csv_files], ignore_index=True)
all_df.loc[all_df['Informativeness_value'].notnull(), 'Cohesion_value'] = 'cohesive'
all_df = all_df.groupby('id').apply(aggregate_group, include_groups=False).apply(pd.Series).reset_index()

cohesive_df = all_df[all_df['Cohesion_value'] == 'cohesive']
incohesive_df = all_df[all_df['Cohesion_value'] == 'incohesive']
min_count = min(len(cohesive_df), len(incohesive_df))

balanced_df = pd.concat([
    cohesive_df.sample(n=min_count, random_state=42),
    incohesive_df.sample(n=min_count, random_state=42),
], ignore_index=True)

print(f"Balanced dataset: {len(balanced_df)} rows (cohesive: {min_count}, incohesive: {min_count})")

In [ ]:
balanced_df.to_csv("balanced_df.csv", index=False)

In [ ]:
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
def run_evaluation(model_configs, balanced_df, guidelines, out_dir, backend="openai", api_client=None):
    """
    Unified LLM-as-a-judge evaluation runner.

    Parameters
    ----------
    model_configs : list[dict]
        Each dict must contain "label" plus:
          - backend="local"     : "repo", optionally "model_kwargs"
          - backend="openai"    : "name", optionally "temperature", "max_tokens"
          - backend="openrouter": "name", optionally "temperature", "max_tokens", "extra_body"
    balanced_df   : pd.DataFrame   Rows with 'source' and 'target' columns.
    guidelines    : str            Task evaluation guidelines (loaded from YAML).
    out_dir       : str | Path     Directory where result CSVs are written.
    backend       : str            "local" | "openai" | "openrouter"
    api_client    : openai.OpenAI  Pre-initialised client (required for openai/openrouter).
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    def _cot_prompt(context, elaboration):
        return (
            f"{guidelines}\n\n"
            "Before providing your evaluation, check the elaboration against:\n"
            "a) Is it relevant to the context and maintains topic continuity?\n"
            "b) Is it free from errors and logical inconsistencies?\n"
            "c) Is it free from misleading examples?\n"
            "d) Does it not repeat parts of the original text?\n\n"
            "Then provide ONLY a JSON object with your evaluation.\n"
            "JSON keys: \"cohesion\" (\"cohesive\" or \"incohesive\") "
            "and \"informativeness\" (integer 1-3, or null if incohesive).\n"
            "No explanation after the JSON.\n\n"
            f"Context: {context}\n"
            f"Elaboration: {elaboration}\n\n"
            "Step-by-step reasoning:\n"
            "1. Connection analysis: [logical flow]\n"
            "2. Topic continuity: [topics align]\n"
            "3. Error check: [accuracy and logic]\n"
            "4. Information value: [what the elaboration adds]\n"
            "5. Overall coherence: [text as a whole]\n"
            "Based on the above, return ONLY the JSON evaluation."
        )

    for m in model_configs:
        label = m["label"]
        print(f"\nEvaluating with {label}...")

        pipe = None
        tokenizer = None

        if backend == "local":
            repo = m["repo"]
            try:
                tokenizer = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
                if tokenizer.pad_token is None:
                    tokenizer.pad_token = tokenizer.eos_token
                pipe = pipeline(
                    "text-generation",
                    model=repo,
                    tokenizer=tokenizer,
                    model_kwargs=m.get("model_kwargs", {"device_map": "auto"}),
                    do_sample=False,
                    repetition_penalty=1.15,
                    max_new_tokens=200,
                    pad_token_id=tokenizer.eos_token_id or tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    skip_special_tokens=True,
                    trust_remote_code=True,
                )
            except Exception as e:
                print(f"  Failed to load {repo}: {e}")
                continue

        results = []
        total = len(balanced_df)

        for idx, row in balanced_df.iterrows():
            if idx % 10 == 0:
                print(f"  {idx + 1}/{total}...")

            context = row["source"]
            elaboration = row["target"]

            try:
                if backend == "local":
                    prompt = create_json_prompt(context, elaboration, guidelines)
                    out = pipe(prompt, return_full_text=False)
                    text = out[0].get("generated_text", str(out[0])) if isinstance(out, list) else str(out)
                    if text.startswith(prompt):
                        text = text[len(prompt):].strip()
                    parsed = normalize_output(text[:300])
                    if parsed["cohesion"] == "incohesive":
                        parsed["informativeness"] = None
                    # Retry if informativeness is missing for a cohesive prediction
                    if parsed["informativeness"] is None and parsed["cohesion"] == "cohesive":
                        try:
                            out2 = pipe(prompt + "\nReturn only the JSON again.", max_new_tokens=50, return_full_text=False)
                            text2 = out2[0].get("generated_text", "") if isinstance(out2, list) else str(out2)
                            p2 = normalize_output(text2)
                            if p2["informativeness"] is not None:
                                parsed["informativeness"] = int(p2["informativeness"])
                            if p2["cohesion"] == "incohesive":
                                parsed["cohesion"] = "incohesive"
                                parsed["informativeness"] = None
                        except Exception:
                            pass
                else:
                    prompt = _cot_prompt(context, elaboration)
                    api_params = {
                        "model": m["name"],
                        "messages": [{"role": "user", "content": prompt}],
                        "temperature": m.get("temperature", 0.0),
                        "max_tokens": m.get("max_tokens", 200),
                    }
                    if m.get("extra_body"):
                        api_params["extra_body"] = m["extra_body"]
                    response = api_client.chat.completions.create(**api_params)
                    text = (response.choices[0].message.content or "").strip()
                    parsed = normalize_output(text)
                    if parsed["cohesion"] == "incohesive":
                        parsed["informativeness"] = None
                    if backend == "openrouter":
                        time.sleep(0.5)

                results.append({
                    "context": context,
                    "elaboration": elaboration,
                    "output": text,
                    "cohesion": parsed["cohesion"],
                    "informativeness": parsed["informativeness"],
                })

            except Exception as e:
                print(f"  Error at row {idx}: {e}")
                results.append({
                    "context": context,
                    "elaboration": elaboration,
                    "output": "",
                    "cohesion": "incohesive",
                    "informativeness": None,
                })

        eval_df = pd.DataFrame(results)
        eval_df["informativeness"] = eval_df["informativeness"].apply(
            lambda x: "" if pd.isna(x) or x is None else str(int(x))
        )
        out_path = out_dir / f"{label}-Evaluation.csv"
        eval_df.to_csv(out_path, index=False)
        cohesive_n = (eval_df["cohesion"] == "cohesive").sum()
        incohesive_n = (eval_df["cohesion"] == "incohesive").sum()
        print(f"  Saved -> {out_path}  (cohesive: {cohesive_n}, incohesive: {incohesive_n})")

        if backend == "local" and pipe is not None:
            try:
                del pipe, tokenizer
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n✓ {backend} evaluations complete.")

In [ ]:
run_evaluation(
    model_configs=hf_models,
    balanced_df=balanced_df,
    guidelines=guidelines,
    out_dir="LLM Evaluations Balanced",
    backend="local",
)

In [ ]:
import openai

openai_client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

run_evaluation(
    model_configs=openai_models,
    balanced_df=balanced_df,
    guidelines=guidelines,
    out_dir="LLM Evaluations Balanced",
    backend="openai",
    api_client=openai_client,
)

In [ ]:
openrouter_client = openai.OpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

run_evaluation(
    model_configs=openrouter_models,
    balanced_df=balanced_df,
    guidelines=guidelines,
    out_dir="LLM Evaluations Balanced",
    backend="openrouter",
    api_client=openrouter_client,
)